In [1]:
import ROOT
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Configuración
year = "2017"
bgr = "signal"
folder = f"plots/{bgr}/{year}/root"

# Crear directorio de salida
output_dir = f"plots/{bgr}/{year}/binning_tables"
Path(output_dir).mkdir(parents=True, exist_ok=True)

# Obtener lista de archivos ROOT
root_files = [f for f in os.listdir(folder) if f.endswith('.root')]

print(f"Procesando {len(root_files)} archivos ROOT...")

for filename in root_files:
    print(f"\nProcesando: {filename}")
    
    file_path = f"{folder}/{filename}"
    
    try:
        # Abrir archivo ROOT
        file = ROOT.TFile.Open(file_path)
        if not file or file.IsZombie():
            continue
        
        # Obtener histogramas
        histograms = []
        for key in file.GetListOfKeys():
            obj = key.ReadObj()
            if obj and obj.InheritsFrom("TH1"):
                histograms.append(obj)
        
        if not histograms:
            file.Close()
            continue
        
        # Preparar datos para el CSV
        bin_data = []
        first_hist = histograms[0]
        nbins = first_hist.GetNbinsX()
        
        # Recorrer todos los bins
        for bin_idx in range(1, nbins + 1):
            bin_info = {
                'bin_number': bin_idx,
                'low_edge': first_hist.GetBinLowEdge(bin_idx),
                'center': first_hist.GetBinCenter(bin_idx),
                'high_edge': first_hist.GetBinLowEdge(bin_idx) + first_hist.GetBinWidth(bin_idx),
                'width': first_hist.GetBinWidth(bin_idx)
            }
            
            # Añadir valores de cada histograma
            for hist in histograms:
                bin_info[hist.GetName()] = hist.GetBinContent(bin_idx)
            
            bin_data.append(bin_info)
        
        # Crear DataFrame
        df = pd.DataFrame(bin_data)
        
        # Añadir fila de totales
        total_row = {'bin_number': 'Total', 'low_edge': '', 'center': '', 'high_edge': '', 'width': ''}
        for hist in histograms:
            total_row[hist.GetName()] = sum(hist.GetBinContent(i+1) for i in range(nbins))
        
        df = pd.concat([df, pd.DataFrame([total_row])], ignore_index=True)
        
        # Guardar como CSV
        csv_filename = f"{output_dir}/{filename.replace('.root', '_binning.csv')}"
        df.to_csv(csv_filename, index=False, float_format='%.6f')
        
        print(f"  ✓ CSV guardado: {csv_filename}")
        print(f"  ✓ Bins: {nbins}, Histogramas: {len(histograms)}")
        print(f"  ✓ Rango: [{first_hist.GetXaxis().GetXmin():.3f}, {first_hist.GetXaxis().GetXmax():.3f}]")
        
        file.Close()
        
    except Exception as e:
        print(f"  ✗ Error con {filename}: {str(e)}")

print(f"\n¡Proceso completado! Tablas de binning guardadas en: {output_dir}")

Welcome to JupyROOT 6.30/04
Procesando 17 archivos ROOT...

Procesando: DYJetsToLNu_signal_tau_2017.root
  ✓ CSV guardado: plots/signal/2017/binning_tables/DYJetsToLNu_signal_tau_2017_binning.csv
  ✓ Bins: 9, Histogramas: 47
  ✓ Rango: [0.000, 300.000]

Procesando: Data_signal_tau_2017.root
  ✓ CSV guardado: plots/signal/2017/binning_tables/Data_signal_tau_2017_binning.csv
  ✓ Bins: 9, Histogramas: 1
  ✓ Rango: [0.000, 300.000]

Procesando: Diboson_signal_tau_2017.root
  ✓ CSV guardado: plots/signal/2017/binning_tables/Diboson_signal_tau_2017_binning.csv
  ✓ Bins: 9, Histogramas: 47
  ✓ Rango: [0.000, 300.000]

Procesando: Higgs_signal_tau_2017.root
  ✓ CSV guardado: plots/signal/2017/binning_tables/Higgs_signal_tau_2017_binning.csv
  ✓ Bins: 9, Histogramas: 47
  ✓ Rango: [0.000, 300.000]

Procesando: QCD_signal_tau_2017.root
  ✓ CSV guardado: plots/signal/2017/binning_tables/QCD_signal_tau_2017_binning.csv
  ✓ Bins: 9, Histogramas: 65
  ✓ Rango: [0.000, 300.000]

Procesando: SignalTau